In [21]:
#Import library
import pandas as pd

#File path
file_path = r"G:\My Drive\DataCamp\sample datasets\xyz_strat_spend_raw_2023-2025.xlsx"

#Check workbook sheets
excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['ERP_Raw_Dump', 'Dim_Budgets']

In [33]:
#Load transaction table from ERP_Raw_Dump sheet
trx = pd.read_excel(
    file_path,
    sheet_name="ERP_Raw_Dump")

# First 5 columns
trx.head()

,RAW_PO_NUMBER,TRANSACTION_DATE,DEPARTMENT_NAME,EXPENSE_CATEGORY,VENDOR_NAME,RAW_AMOUNT,GL_CODE,TAX_RATE,APPROVAL_STATUS,INVOICE_REF,SYSTEM_COMMENTS
0,PO-2024-2001,26-Oct-25,it operations,Office Rent,CloudSaaS Corp,"PHP282,706.32",GL-600300,12%,REJECTED,INV-9984,System Auto-Generated
1,PO-2024-2002,02-Oct-25,Facilities and Ops,Travel & Entertainment,Prime Space Realty,"PHP115,160.76",GL-500100,12%,APPROVED,NaN,System Auto-Generated
2,,01-Oct-25,MARKETING,Cloud Infrastructure,TechDistributors Inc,"PHP104,166.79",GL-500200,12%,PENDING,INV-8207,System Auto-Generated
3,PO-2024-2004,28-Nov-24,IT Operations,Office Supplies,Prime Space Realty,"PHP90,582.74",GL-500200,12%,PENDING,INV-6978,Urgent operational requirement!
4,PO-2024-2005,05-Jul-23,Facilities & Ops,Office Supplies,Prime Space Realty,"PHP111,666.30",GL-500100,12%,REJECTED,INV-7440,System Auto-Generated


In [30]:
#Load budget table from 'Dim_Budgets'
budget = pd.read_excel(
    file_path,
    sheet_name="Dim_Budgets")

#First 5 columns
budget.head()

,Fiscal_Year,Department,Allocated_Budget_PHP
0,2023,IT Operations,6500000
1,2023,Marketing & Sales,5200000
2,2023,HR & Admin,4100000
3,2023,Facilities & Ops,5800000
4,2024,IT Operations,6890000


In [36]:
# Data Quality Assessment
#1.1 Check missing values

missing_values = trx.isnull().sum()
missing_values

RAW_PO_NUMBER         0
TRANSACTION_DATE      0
DEPARTMENT_NAME      53
EXPENSE_CATEGORY      0
VENDOR_NAME           0
RAW_AMOUNT           20
GL_CODE               0
TAX_RATE              0
APPROVAL_STATUS       0
INVOICE_REF         149
SYSTEM_COMMENTS       0
dtype: int64

In [37]:
missing_values = budget.isnull().sum()
missing_values

Fiscal_Year             0
Department              0
Allocated_Budget_PHP    0
dtype: int64

In [41]:
#1.2 Missing values summary
missing_summary = pd.DataFrame({
    "Missing_Count": trx.isnull().sum(),
    "Missing_Percentage": trx.isnull().mean() * 100})

missing_summary.sort_values(
    "Missing_Count", ascending=False)

,Missing_Count,Missing_Percentage
INVOICE_REF,149,11.595331
DEPARTMENT_NAME,53,4.124514
RAW_AMOUNT,20,1.556420
TRANSACTION_DATE,0,0.000000
RAW_PO_NUMBER,0,0.000000
VENDOR_NAME,0,0.000000
EXPENSE_CATEGORY,0,0.000000
GL_CODE,0,0.000000
TAX_RATE,0,0.000000
APPROVAL_STATUS,0,0.000000


In [42]:
#2.1 Check duplicates
trx.duplicated().sum()

np.int64(35)

In [43]:
budget.duplicated().sum()

np.int64(0)

In [48]:
#2.2 Display exact duplicate records from transaction records
duplicates = trx[trx.duplicated(keep=False)]

duplicates.sort_values(
    by=trx.columns.tolist())

,RAW_PO_NUMBER,TRANSACTION_DATE,DEPARTMENT_NAME,EXPENSE_CATEGORY,VENDOR_NAME,RAW_AMOUNT,GL_CODE,TAX_RATE,APPROVAL_STATUS,INVOICE_REF,SYSTEM_COMMENTS
886,,14-Dec-23,IT Operations,Office Supplies,CloudSaaS Corp,"PHP310,041.08",GL-500200,12%,Approved,INV-7667,System Auto-Generated
1279,,14-Dec-23,IT Operations,Office Supplies,CloudSaaS Corp,"PHP310,041.08",GL-500200,12%,Approved,INV-7667,System Auto-Generated
48,,26-Mar-24,MARKETING,Software Licenses,GlobalMedia Ads,"PHP80,512.44",GL-500100,12%,PENDING,INV-8552,System Auto-Generated
1283,,26-Mar-24,MARKETING,Software Licenses,GlobalMedia Ads,"PHP80,512.44",GL-500100,12%,PENDING,INV-8552,System Auto-Generated
1227,,30-Aug-24,MARKETING,Travel & Entertainment,TechDistributors Inc,"PHP213,470.00",GL-500200,12%,PENDING,INV-4062,Urgent operational requirement!
...,...,...,...,...,...,...,...,...,...,...,...
1270,PO-2024-3146,11/22/2024,Facilities & Ops,Office Rent,CloudSaaS Corp,"PHP205,611.74",GL-500100,12%,Approved,INV-8093,Urgent operational requirement!
1175,PO-2024-3176,10/09/2025,IT Operations,Travel & Entertainment,Prime Space Realty,"PHP65,136.42",GL-500200,12%,APPROVED,INV-8333,System Auto-Generated
1260,PO-2024-3176,10/09/2025,IT Operations,Travel & Entertainment,Prime Space Realty,"PHP65,136.42",GL-500200,12%,APPROVED,INV-8333,System Auto-Generated
1249,PO-2024-3250,09-May-25,NaN,Office Rent,PaperCorp Supplies,"PHP137,630.15",GL-600300,0%,REJECTED,INV-6223,System Auto-Generated


In [59]:
## DATA QUALITY ACTIONS ###

# Standardize Department Names
department_mapping = {
    'it operations': 'IT Operations',
    'IT Operations': 'IT Operations',
    'MARKETING': 'Marketing & Sales',
    'Marketing & Sales': 'Marketing & Sales',
    'Facilities and Ops': 'Facilities & Ops',
    'Facilities & Ops': 'Facilities & Ops',
    'HR & Admin': 'HR & Admin'}

trx['DEPARTMENT_NAME'] = trx['DEPARTMENT_NAME'].map(
    department_mapping).fillna('Unassigned')

# Check standardized departments
print("--- Standardized Departments ---")
print(trx['DEPARTMENT_NAME'].value_counts())

--- Standardized Departments ---
DEPARTMENT_NAME
Facilities & Ops     367
Marketing & Sales    339
IT Operations        326
HR & Admin           168
Unassigned            50
Name: count, dtype: int64


In [62]:
# Standardize Approval Status
trx['APPROVAL_STATUS'] = (
    trx['APPROVAL_STATUS']
    .astype(str)
    .str.strip()
    .str.upper())

# Keep only approved transactions
trx = trx[
    trx['APPROVAL_STATUS'] == 'APPROVED'
].copy()

# Check current status
print("--- Current Approval Status ---")
print(trx['APPROVAL_STATUS'].value_counts())

--- Current Approval Status ---
APPROVAL_STATUS
APPROVED    613
Name: count, dtype: int64


In [63]:
# Drop exact duplicate rows, keeping the first occurrence
trx = trx.drop_duplicates(
    keep='first'
)

# Reset the index
trx = trx.reset_index(drop=True)

print("Remaining rows:", len(trx))
print("Remaining exact duplicates:", trx.duplicated().sum())

Remaining rows: 613
Remaining exact duplicates: 0


In [66]:
# Standardize transaction dates
trx["TRANSACTION_DATE"] = pd.to_datetime(
    trx["TRANSACTION_DATE"].astype(str).str.strip(),
    errors="coerce")

# Check the result
print(trx["TRANSACTION_DATE"].head())
print("\nData type:")
print(trx["TRANSACTION_DATE"].dtype)

0   2025-10-02
1   2025-09-08
2   2025-10-24
3   2025-10-16
4   2023-05-23
Name: TRANSACTION_DATE, dtype: datetime64[ns]

Data type:
datetime64[ns]


In [71]:
# Standardize raw amounts
trx["RAW_AMOUNT"] = (
    trx["RAW_AMOUNT"]
    .astype(str)
    .str.strip()
    .str.replace("PHP", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Convert to numeric
trx["RAW_AMOUNT"] = pd.to_numeric(
    trx["RAW_AMOUNT"],
    errors="coerce")

print(trx["RAW_AMOUNT"].head(10))
print("\nData type:")
print(trx["RAW_AMOUNT"].dtype)

0    115160.76
1    186051.69
2    113149.23
3    206325.42
4     57783.36
5     51244.28
6     92128.53
7     36320.02
8     50577.43
9     17764.09
Name: RAW_AMOUNT, dtype: float64

Data type:
float64


In [77]:
# Standardize department names
department_mapping = {
    "it operations": "IT Operations",
    "IT Operations": "IT Operations",
    "MARKETING": "Marketing & Sales",
    "Marketing & Sales": "Marketing & Sales",
    "Facilities and Ops": "Facilities & Ops",
    "Facilities & Ops": "Facilities & Ops",
    "HR & Admin": "HR & Admin"
}

trx["DEPARTMENT_NAME"] = (
    trx["DEPARTMENT_NAME"]
    .map(department_mapping)
    .fillna("Unassigned")
)

print(trx["DEPARTMENT_NAME"].value_counts())

DEPARTMENT_NAME
Facilities & Ops     190
Marketing & Sales    161
IT Operations        155
HR & Admin            80
Unassigned            27
Name: count, dtype: int64


In [76]:
# Standardize expense category formatting
trx["EXPENSE_CATEGORY"] = (
    trx["EXPENSE_CATEGORY"]
    .astype(str)
    .str.strip()
)

# Check unique categories
print(trx["EXPENSE_CATEGORY"].value_counts())

EXPENSE_CATEGORY
Software Licenses         105
Hardware Procurement      102
Office Supplies            87
Digital Advertising        84
Office Rent                82
Cloud Infrastructure       80
Travel & Entertainment     73
Name: count, dtype: int64


In [78]:
# Final validation of standardized fields
print("--- Data Types ---")
print(trx[
    [
        "TRANSACTION_DATE",
        "RAW_AMOUNT",
        "DEPARTMENT_NAME",
        "EXPENSE_CATEGORY"
    ]
].dtypes)

print("\n--- Departments ---")
print(trx["DEPARTMENT_NAME"].unique())

print("\n--- Expense Categories ---")
print(trx["EXPENSE_CATEGORY"].unique())

print("\n--- Missing Values ---")
print(trx[
    [
        "TRANSACTION_DATE",
        "RAW_AMOUNT",
        "DEPARTMENT_NAME",
        "EXPENSE_CATEGORY"
    ]
].isnull().sum())

--- Data Types ---
TRANSACTION_DATE    datetime64[ns]
RAW_AMOUNT                 float64
DEPARTMENT_NAME             object
EXPENSE_CATEGORY            object
dtype: object

--- Departments ---
['Facilities & Ops' 'IT Operations' 'Marketing & Sales' 'Unassigned'
 'HR & Admin']

--- Expense Categories ---
['Travel & Entertainment' 'Office Supplies' 'Digital Advertising'
 'Office Rent' 'Cloud Infrastructure' 'Hardware Procurement'
 'Software Licenses']

--- Missing Values ---
TRANSACTION_DATE     0
RAW_AMOUNT          11
DEPARTMENT_NAME      0
EXPENSE_CATEGORY     0
dtype: int64


In [81]:
## Exporting of cleaned source file ##
# Export cleaned data as xlsx file
output_file = r"G:\My Drive\DataCamp\sample datasets\axiom_strat_spend_cleaned_2023-2025.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    
    # Cleaned transaction data
    trx.to_excel(
        writer,
        sheet_name="ERP_Cleaned",
        index=False
    )
    
    # Budget data
    budget.to_excel(
        writer,
        sheet_name="Dim_Budgets",
        index=False
    )